In [51]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import uproot
import os
import sys

sys.path.append('/exp/sbnd/data/users/aantonak/SBNDWireModFitting/')

In [46]:
os.system("ls /exp/sbnd/data/users/aantonak/WireMod_GEN2_DATA/")

GEN2_V0_Projections
GEN2_V0_Splines
WIRE_Gen2_Angles_2D_Data_Full.root


0

In [47]:
f = "/exp/sbnd/data/users/aantonak/WireMod_GEN2_DATA/WIRE_Gen2_Angles_2D_Data_Full.root"

In [48]:
fu = uproot.open(f)

In [49]:
print(fu.keys())

['hHit0;1', 'hTrack0;1', 'hHit1;1', 'hTrack1;1', 'hHit2;1', 'hTrack2;1', 'hHit3;1', 'hTrack3;1', 'hHit4;1', 'hTrack4;1', 'hHit5;1', 'hTrack5;1']


In [50]:
# Locate THnSparse/TH2D named hTrack0 (handles cycle suffixes like ;1)
import numpy as np
import matplotlib.pyplot as plt

# find a matching key in the file (be robust to bytes/strings and cycle suffixes)
hist_key = None
for key in fu.keys():
    k = key.decode('utf-8') if isinstance(key, (bytes, bytearray)) else str(key)
    if 'hTrack0' in k:
        hist_key = k.split(';')[0]
        break
if hist_key is None:
    raise KeyError(f'hTrack0 not found in file. Available keys:{fu.keys()}')

hist = fu[hist_key]
print('Using histogram object:', hist)
print('Type:', type(hist))
# show a short dir to help debugging when attributes are missing
print('dir(hist) sample:', [n for n in dir(hist) if not n.startswith('_')][:60])

# Try to extract numeric arrays. Support TH2-like and THnSparse-like objects.
edges = None
values = None

# Prefer to_numpy when available and working
if hasattr(hist, 'to_numpy'):
    try:
        res = hist.to_numpy()
        if isinstance(res, tuple) and len(res) == 3:
            edges_x, edges_y, values = res
            edges = (np.asarray(edges_x), np.asarray(edges_y))
        else:
            values = np.asarray(res)
    except Exception as e:
        print('hist.to_numpy() exists but raised:', repr(e))
        values = None
else:
    print('hist.to_numpy() not available; falling back')

# Fallback: try .values() and axes edges
if values is None:
    if hasattr(hist, 'values'):
        try:
            values = np.asarray(hist.values())
        except Exception as e:
            print('hist.values() failed:', repr(e))
            values = None

if values is None:
    # Try to access member arrays from uproot model objects (e.g. THnSparse internals)
    if hasattr(hist, 'member'):
        try:
            # common member names: fArray (contents), fAxes/fAxes (edges held on axes objects)
            try:
                arr = hist.member('fArray')
                values = np.asarray(arr)
                print('Retrieved values from member fArray')
            except Exception:
                # last resort: try iterating numeric members
                members = [m for m in getattr(hist, 'members', lambda: [])() if m is not None]
                print('hist.members() sample length:', len(members) if hasattr(hist, 'members') else 'n/a')
        except Exception as e:
            print('Could not extract values from member()', repr(e))

if values is None:
    raise RuntimeError('Could not extract histogram values from object; see diagnostics above')

# If this is a multidimensional sparse histogram (>=3 axes), project (sum) over axes 2..N-1 to get a 2D histogram.
if values.ndim >= 3:
    print(f'Detected {values.ndim}D sparse histogram; projecting (summing) over extra axes to produce 2D')
    # sum over axes beyond the first two
    sum_axes = tuple(range(2, values.ndim))
    Z = values.sum(axis=sum_axes)
    # edges: prefer the first two axes
    if edges is None:
        # try to get edges from hist.axes if available
        try:
            edges_x = np.asarray(hist.axes[0].edges())
            edges_y = np.asarray(hist.axes[1].edges())
            edges = (edges_x, edges_y)
        except Exception as e:
            print('Could not get axes.edges():', repr(e))
            # fallback to integer bins
            nx = Z.shape[1]
            ny = Z.shape[0]
            edges = (np.arange(nx+1), np.arange(ny+1))
else:
    # values is 1D or 2D already
    Z = values
    if Z.ndim == 1:
        # if 1D but axes indicate 2D binning, attempt to reshape using edges or axis bin counts
        if edges is not None:
            nx = edges[0].size - 1
            ny = edges[1].size - 1
            if Z.size == nx * ny:
                Z = Z.reshape((ny, nx))
        else:
            # try to get bin counts from hist.axes
            try:
                nx = int(getattr(hist.axes[0], 'nbins', hist.axes[0].size))
                ny = int(getattr(hist.axes[1], 'nbins', hist.axes[1].size))
                if Z.size == nx * ny:
                    Z = Z.reshape((ny, nx))
            except Exception:
                pass
    if edges is None:
        # try to infer edges
        try:
            edges_x = np.asarray(hist.axes[0].edges())
            edges_y = np.asarray(hist.axes[1].edges())
            edges = (edges_x, edges_y)
        except Exception as e:
            print('Could not infer edges from axes:', repr(e))
            nx = Z.shape[1]
            ny = Z.shape[0]
            edges = (np.arange(nx+1), np.arange(ny+1))

edges_x, edges_y = edges

# Ensure arrays are numpy arrays
edges_x = np.asarray(edges_x)
edges_y = np.asarray(edges_y)
Z = np.asarray(Z)

print('edges_x.shape, edges_y.shape, Z.shape ->', edges_x.shape, edges_y.shape, Z.shape)

# pcolormesh expects Z shape (Ny, Nx) with X length Nx+1 and Y length Ny+1
nx = edges_x.size - 1
ny = edges_y.size - 1
if Z.shape != (ny, nx):
    if Z.shape == (nx, ny):
        Z = Z.T
    else:
        if Z.size == nx * ny:
            Z = Z.reshape((ny, nx))
        else:
            raise ValueError(f'Unexpected histogram shape: {Z.shape}, expected ({ny},{nx})')

# Try to obtain axis labels if available
xlabel = 'X'
ylabel = 'Y'
try:
    ax0 = hist.axes[0]
    ax1 = hist.axes[1]
    xlabel = getattr(ax0, 'label', xlabel) or xlabel
    ylabel = getattr(ax1, 'label', ylabel) or ylabel
except Exception:
    pass

X, Y = np.meshgrid(edges_x, edges_y)
fig, ax = plt.subplots(figsize=(8,6))
pcm = ax.pcolormesh(X, Y, Z, shading='auto')
fig.colorbar(pcm, ax=ax, label='Counts')
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_title(hist_key)
plt.show()


Using histogram object: <THnSparseT<TArrayD> (version 1) at 0x7f6d8ab01a80>
Type: <class 'uproot.dynamic.Model_THnSparseT_3c_TArrayD_3e__v1'>
dir(hist) sample: ['all_members', 'awkward_form', 'base', 'base_names_versions', 'bases', 'behaviors', 'check_numbytes', 'class_code', 'class_flags', 'class_rawstreamers', 'class_streamer', 'class_version', 'classname', 'close', 'closed', 'concrete', 'cursor', 'empty', 'encoded_classname', 'file', 'has_member', 'hook_after_read_members', 'hook_before_postprocess', 'hook_before_read', 'hook_before_read_members', 'instance_version', 'is_instance', 'is_memberwise', 'member', 'member_names', 'members', 'num_bytes', 'parent', 'postprocess', 'read', 'read_member_n', 'read_members', 'read_numbytes_version', 'serialize', 'strided_interpretation', 'to_pyroot', 'to_writable', 'tojson', 'writable']
hist.to_numpy() not available; falling back
Could not extract values from member() TypeError("'dict' object is not callable")


RuntimeError: Could not extract histogram values from object; see diagnostics above